In [ ]:
import papermill as pm
from jaxcmr.helpers import find_project_root
from IPython.display import Image, display
import os

# Render Model Fitting: Full eCMR Models

Orchestrate subject-level fitting notebooks for the full eCMR model family.


In [ ]:
from lpp_ecmr.fitting_config import ECMR_BASE_PARAMS
from lpp_ecmr.full_ecmr_registry import ECMR_MODELS

base_params = ECMR_BASE_PARAMS


In [ ]:
varied_parameters = [m for m in ECMR_MODELS if m.get("enabled")]
print(f"{len(varied_parameters)} models enabled")


In [ ]:
project_root = find_project_root()
workspace = os.path.dirname(project_root)
template = os.path.join(workspace, "jaxcmr", "templates", "fitting.ipynb")
rendered_dir = os.path.join(project_root, "analyses", "rendered")
figure_dir = os.path.join(project_root, "figures", "fitting")
os.makedirs(rendered_dir, exist_ok=True)

for partial_params in varied_parameters:

    params = base_params.copy() | {
        k: v for k, v in partial_params.items() if k != "enabled"
    }

    data_tag = params["data_tag"]
    model_name = params["model_name"]
    base_run_tag = params["base_run_tag"]
    best_of = params["best_of"]
    fit_tag = f"{base_run_tag}_best_of_{best_of}"
    max_subjects = params["max_subjects"]
    if max_subjects:
        fit_tag += f"_nsubs_{max_subjects}"

    template_params = params | {
        "trial_query": params["trial_query"],
        "target_directory": "",
        "figure_dir": "figures/fitting",
        "figure_str": f"{data_tag}_{model_name}_{fit_tag}",
        "display_iterations": False,
    }

    output_path = os.path.join(
        rendered_dir,
        f"fitting_{data_tag}_{model_name}_{fit_tag}.ipynb",
    )
    print(output_path)

    pm.execute_notebook(
        template,
        output_path,
        parameters=template_params,
        progress_bar=True,
        prepare_only=True,
    )

    for analysis_cfg in template_params["single_analysis_configs"]:
        for dataset_label in ["data", "sim"]:
            figure_name = (
                f"{template_params['figure_str']}_{analysis_cfg['figure_suffix']}_{dataset_label}.png"
            )
            figure_path = os.path.join(figure_dir, figure_name)
            print(f"![]({figure_path})")
            if os.path.exists(figure_path):
                display(Image(filename=figure_path, width=600))

    for analysis_cfg in template_params["comparison_analysis_configs"]:
        figure_name = (
            f"{template_params['figure_str']}_{analysis_cfg['figure_suffix']}.png"
        )
        figure_path = os.path.join(figure_dir, figure_name)
        print(f"![]({figure_path})")
        if os.path.exists(figure_path):
            display(Image(filename=figure_path, width=600))